# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.  
> **Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

---

### Rules to keep in mind
1. Every bar chart **must have a zero baseline** — no exceptions
2. Every chart **must have an insight title**, not a topic title
3. Aim for **professional quality** — clean background, readable font, no clutter
4. Horizontal bars for long category names

---

## Setup — Run this cell first

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']

print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())

---
## Task 1 — Regional Comparison Bar Chart

**Insight title:** *Western Europe leads global happiness by a wide margin — Sub-Saharan Africa lags 3 points behind*

**What to build:** Horizontal bar chart — average happiness score by region, sorted descending.

In [ ]:
# Task 1 — Regional Average Happiness (Horizontal Bar Chart)
# -----------------------------------------------------------

# Step 1: Compute regional averages, sorted ascending (so top = happiest)
region_avg = (
    df.groupby('Region')['Happiness_Score']
    .mean()
    .reset_index()
    .sort_values('Happiness_Score', ascending=True)   # ascending for horiz. bar
)

# Step 2: Colour scale — map score to a sequential purple palette
min_s, max_s = region_avg['Happiness_Score'].min(), region_avg['Happiness_Score'].max()
norm = (region_avg['Happiness_Score'] - min_s) / (max_s - min_s)

# Step 3: Build chart
FONT = 'Inter, Arial, sans-serif'
BG   = '#0f0f1a'
GRID = 'rgba(255,255,255,0.07)'

fig1 = go.Figure()

fig1.add_trace(go.Bar(
    x=region_avg['Happiness_Score'],
    y=region_avg['Region'],
    orientation='h',
    marker=dict(
        color=norm,
        colorscale=[
            [0.0,  '#f87171'],   # low  — warm red
            [0.5,  '#a78bfa'],   # mid  — purple
            [1.0,  '#6ee7b7'],   # high — mint green
        ],
        line=dict(width=0),
    ),
    text=region_avg['Happiness_Score'].round(2),
    textposition='outside',
    textfont=dict(color='white', size=12, family=FONT),
    hovertemplate='<b>%{y}</b><br>Avg Score: %{x:.2f}<extra></extra>',
))

# Global average reference line
global_avg = df['Happiness_Score'].mean()
fig1.add_vline(
    x=global_avg,
    line_dash='dash',
    line_color='rgba(255,255,255,0.4)',
    line_width=1.5,
    annotation_text=f'Global avg {global_avg:.2f}',
    annotation_position='top right',
    annotation_font=dict(color='rgba(255,255,255,0.5)', size=11, family=FONT),
)

fig1.update_layout(
    title=dict(
        text=(
            '<b>Western Europe leads global happiness by a wide margin</b><br>'
            '<span style="font-size:13px; color:#a0a0b0;">'
            'Sub-Saharan Africa scores ~3 points lower — the largest regional gap in 2023'
            '</span>'
        ),
        font=dict(size=17, color='white', family=FONT),
        x=0.02, xanchor='left', y=0.97,
    ),
    xaxis=dict(
        title='Average Happiness Score',
        range=[0, 8.5],           # ZERO BASELINE
        tickfont=dict(color='rgba(255,255,255,0.6)', family=FONT),
        titlefont=dict(color='rgba(255,255,255,0.6)', family=FONT),
        gridcolor=GRID,
        zeroline=True,
        zerolinecolor='rgba(255,255,255,0.2)',
        zerolinewidth=1.5,
    ),
    yaxis=dict(
        tickfont=dict(color='white', size=12, family=FONT),
        gridcolor='rgba(0,0,0,0)',
    ),
    plot_bgcolor=BG,
    paper_bgcolor=BG,
    margin=dict(l=200, r=80, t=100, b=60),
    height=460,
    showlegend=False,
)

fig1.show()

---
## Task 2 — Top 8 vs. Bottom 8: A Contrast Story

**Insight title:** *The happiness gap between the world's top and bottom countries is nearly 5 points — an almost unbridgeable divide*

**What to build:** Horizontal bar chart — top 8 + bottom 8 countries, colour-coded, with global avg line.

In [ ]:
# Task 2 — Top 8 vs Bottom 8 Contrast Bar Chart
# -----------------------------------------------

top8    = df.nlargest(8,  'Happiness_Score').copy(); top8['Group']    = 'Top 8 Happiest'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy(); bottom8['Group'] = 'Bottom 8 Least Happy'

combined   = pd.concat([bottom8, top8]).sort_values('Happiness_Score', ascending=True)
global_avg = df['Happiness_Score'].mean()

# Colour map
color_map = {
    'Top 8 Happiest':      '#6ee7b7',   # mint
    'Bottom 8 Least Happy': '#f87171',  # coral red
}
bar_colors = combined['Group'].map(color_map)

FONT = 'Inter, Arial, sans-serif'
BG   = '#0f0f1a'
GRID = 'rgba(255,255,255,0.07)'

fig2 = go.Figure()

# One trace per group (enables clean legend)
for group, color in color_map.items():
    sub = combined[combined['Group'] == group]
    fig2.add_trace(go.Bar(
        x=sub['Happiness_Score'],
        y=sub['Country'],
        orientation='h',
        name=group,
        marker=dict(color=color, line=dict(width=0)),
        text=sub['Happiness_Score'].round(2),
        textposition='outside',
        textfont=dict(color='white', size=11, family=FONT),
        hovertemplate='<b>%{y}</b><br>Score: %{x:.3f}<extra></extra>',
    ))

# Global average reference line
fig2.add_vline(
    x=global_avg,
    line_dash='dot',
    line_color='rgba(255,215,0,0.6)',
    line_width=2,
    annotation_text=f'Global avg {global_avg:.2f}',
    annotation_position='top right',
    annotation_font=dict(color='rgba(255,215,0,0.8)', size=11, family=FONT),
)

# Gap annotation between groups
top_min  = top8['Happiness_Score'].min()
bot_max  = bottom8['Happiness_Score'].max()
gap      = top_min - bot_max

# Separator line between the two groups
fig2.add_vline(
    x=(top_min + bot_max) / 2,
    line_dash='solid',
    line_color='rgba(255,255,255,0.12)',
    line_width=1,
)

fig2.update_layout(
    title=dict(
        text=(
            '<b>The happiness gap between the world\'s top & bottom nations is nearly 5 points</b><br>'
            '<span style="font-size:13px; color:#a0a0b0;">'
            f'Finland (7.80) vs. Afghanistan (1.86) — a divide of {df["Happiness_Score"].max() - df["Happiness_Score"].min():.2f} points'
            '</span>'
        ),
        font=dict(size=17, color='white', family=FONT),
        x=0.02, xanchor='left', y=0.98,
    ),
    xaxis=dict(
        title='Happiness Score (Ladder Score)',
        range=[0, 9.0],           # ZERO BASELINE
        tickfont=dict(color='rgba(255,255,255,0.6)', family=FONT),
        titlefont=dict(color='rgba(255,255,255,0.6)', family=FONT),
        gridcolor=GRID,
        zeroline=True,
        zerolinecolor='rgba(255,255,255,0.2)',
        zerolinewidth=1.5,
    ),
    yaxis=dict(
        tickfont=dict(color='white', size=12, family=FONT),
        gridcolor='rgba(0,0,0,0)',
    ),
    legend=dict(
        font=dict(color='white', family=FONT, size=12),
        bgcolor='rgba(255,255,255,0.07)',
        bordercolor='rgba(255,255,255,0.15)',
        borderwidth=1,
        x=0.72, y=0.05,
    ),
    barmode='overlay',
    plot_bgcolor=BG,
    paper_bgcolor=BG,
    margin=dict(l=140, r=90, t=110, b=60),
    height=560,
)

fig2.show()

---
## Stretch Goal — Grouped Bar: GDP vs Freedom across Key Regions

In [ ]:
# Stretch — Grouped Bar Chart
# ----------------------------

target_regions = [
    'Western Europe', 'Latin America and Caribbean',
    'East Asia', 'Sub-Saharan Africa', 'South Asia'
]

sub = df[df['Region'].isin(target_regions)].copy()
grouped = sub.groupby('Region')[['GDP','Freedom']].mean().reset_index()
grouped_m = grouped.melt(id_vars='Region', var_name='Factor', value_name='Score')

FONT = 'Inter, Arial, sans-serif'
BG   = '#0f0f1a'
GRID = 'rgba(255,255,255,0.07)'

fig3 = px.bar(
    grouped_m,
    x='Region',
    y='Score',
    color='Factor',
    barmode='group',
    color_discrete_map={'GDP': '#60a5fa', 'Freedom': '#f9a8d4'},
    text=grouped_m['Score'].round(2),
)

fig3.update_traces(textposition='outside', textfont=dict(color='white', size=11, family=FONT))

fig3.update_layout(
    title=dict(
        text=(
            '<b>High GDP does not guarantee high freedom — South Asia shows the largest gap</b><br>'
            '<span style="font-size:13px; color:#a0a0b0;">'
            'GDP and Freedom to choose diverge sharply across regions in 2023'
            '</span>'
        ),
        font=dict(size=17, color='white', family=FONT),
        x=0.02, xanchor='left',
    ),
    xaxis=dict(
        title='Region',
        tickfont=dict(color='white', size=11, family=FONT),
        titlefont=dict(color='rgba(255,255,255,0.6)', family=FONT),
        gridcolor='rgba(0,0,0,0)',
    ),
    yaxis=dict(
        title='Score',
        range=[0, 12],
        tickfont=dict(color='rgba(255,255,255,0.6)', family=FONT),
        titlefont=dict(color='rgba(255,255,255,0.6)', family=FONT),
        gridcolor=GRID,
        zeroline=True,
        zerolinecolor='rgba(255,255,255,0.2)',
        zerolinewidth=1.5,
    ),
    legend=dict(
        font=dict(color='white', family=FONT, size=12),
        bgcolor='rgba(255,255,255,0.07)',
        bordercolor='rgba(255,255,255,0.15)',
        borderwidth=1,
    ),
    plot_bgcolor=BG,
    paper_bgcolor=BG,
    margin=dict(l=60, r=60, t=110, b=80),
    height=480,
)

fig3.show()